# 01 - Exploratory Data Analysis: Telco Customer Churn

Notebook interaktif pendamping `src/eda.py` (versi script yang menyimpan
figure dan ringkasan ke `reports/`). Jalankan sel per sel untuk eksplorasi.

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import load_raw, TARGET

df = load_raw()
df.shape

## 1. Distribusi target

Dataset ini tidak seimbang: sekitar seperempat pelanggan churn.
Karena itu akurasi bukan metrik utama, gunakan PR-AUC dan F1.

In [ ]:
vc = df[TARGET].value_counts(normalize=True).round(4)
print(vc)
vc.plot.bar(color=["#2a9d8f", "#e76f51"], title="Target distribution")
plt.ylabel("Proportion");

## 2. Missing values

`TotalCharges` mengandung string kosong pada pelanggan dengan tenure 0
(sudah dikoersi menjadi NaN di `data_loader`).

In [ ]:
miss = df.isna().sum()
print(miss[miss > 0])
print("tenure==0 untuk baris NaN:", (df.loc[df["TotalCharges"].isna(), "tenure"] == 0).all())

## 3. Fitur numerik vs churn

In [ ]:
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    fig, ax = plt.subplots()
    for label, color in (("No", "#2a9d8f"), ("Yes", "#e76f51")):
        ax.hist(df.loc[df[TARGET] == label, col].dropna(), bins=24,
                alpha=0.6, label=f"Churn={label}", color=color)
    ax.set_title(f"{col} by churn"); ax.legend(); plt.show()

## 4. Churn rate per kategori

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    rates = df.groupby(col, observed=True)[TARGET].apply(lambda s: (s == "Yes").mean())
    print(f"--- {col} ---")
    print(rates.sort_values(ascending=False).round(3).to_string())
    print()

## 5. Temuan kunci

Ringkasan angka dan figure tersimpan otomatis oleh `src/eda.py` ke
`reports/eda_summary.md` dan `reports/figures/`. Kesimpulan utama:

- Target imbalanced, evaluasi memakai PR-AUC/F1.
- Missing hanya di `TotalCharges` (tenure 0), aman diimputasi.
- Tenure pendek + kontrak month-to-month = churn tertinggi.